# Lesson 4.2 - GPU Serving Stack With vLLM

**Level: Intermediate**

## Learning Objectives
- Deploy NVIDIA device plugin for GPU resource discovery.
- Deploy vLLM with GPU scheduling constraints.
- Expose vLLM internally using a ClusterIP service.

**Project file(s) produced:** `gitops/nvidia-device-plugin/daemonset.yaml`, `gitops/nvidia-device-plugin/kustomization.yaml`, `gitops/vllm-qwen/deployment.yaml`, `gitops/vllm-qwen/service.yaml`, `gitops/vllm-qwen/kustomization.yaml`

In [ ]:
%%writefile gitops/nvidia-device-plugin/daemonset.yaml
apiVersion: apps/v1
kind: DaemonSet
metadata:
  name: nvidia-device-plugin-daemonset
  namespace: kube-system
spec:
  selector:
    matchLabels:
      name: nvidia-device-plugin-ds
  template:
    metadata:
      labels:
        name: nvidia-device-plugin-ds
    spec:
      tolerations:
        - key: nvidia.com/gpu
          operator: Equal
          value: present
          effect: NoSchedule
      containers:
        - image: nvcr.io/nvidia/k8s-device-plugin:v0.17.0
          name: nvidia-device-plugin-ctr
      nodeSelector:
        agentpool: gpu

In [ ]:
%%writefile gitops/vllm-qwen/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: vllm
  namespace: ai-demos
spec:
  replicas: 1
  selector:
    matchLabels:
      app: vllm
  template:
    metadata:
      labels:
        app: vllm
    spec:
      containers:
        - name: vllm
          image: vllm/vllm-openai:latest
          args:
            - "Qwen/Qwen2.5-7B-Instruct-AWQ"
            - "--quantization"
            - "awq_marlin"
            - "--served-model-name"
            - "qwen2.5-7b"
          resources:
            requests:
              nvidia.com/gpu: "1"
      tolerations:
        - key: "nvidia.com/gpu"
          operator: "Equal"
          value: "present"
          effect: "NoSchedule"

In [ ]:
%%writefile gitops/vllm-qwen/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: vllm-serv
  namespace: ai-demos
spec:
  type: ClusterIP
  selector:
    app: vllm
  ports:
    - name: http
      port: 8000
      targetPort: 8000

In [ ]:
%%writefile gitops/nvidia-device-plugin/kustomization.yaml
apiVersion: kustomize.config.k8s.io/v1beta1
kind: Kustomization
resources:
  - daemonset.yaml

%%writefile gitops/vllm-qwen/kustomization.yaml
apiVersion: kustomize.config.k8s.io/v1beta1
kind: Kustomization
resources:
  - deployment.yaml
  - service.yaml

## Exercise
Swap vLLM model to another compatible Hugging Face model and update resource assumptions.

In [ ]:
# TODO: Change model arg in vllm deployment.
# TODO: If model is gated, define HF token flow via Key Vault CSI.
print('Check readiness with: kubectl get pods -n ai-demos -w')

---
**Next:** [Lesson 4.3 - CPU Serving, HPA, And OpenWebUI](./lesson-4.3-cpu-serving-hpa-and-openwebui.ipynb)